# Parameter-Efficient Fine-Tuning of Clinical LLMs for Automated Clinical Information Extraction from MIMIC-III

**Platform:** NVIDIA DGX Spark &nbsp;|&nbsp; **Data:** MIMIC-III `NOTEEVENTS` (DuckDB) &nbsp;|&nbsp; **Teacher / zero-shot model:** `nvidia/Qwen3.6-35B-A3B-NVFP4` served by vLLM &nbsp;|&nbsp; **Student (LoRA):** `Qwen/Qwen2.5-3B-Instruct`

This notebook implements the end-to-end pipeline described in the research synopsis:

| Synopsis objective | Notebook section |
|---|---|
| 1. Preprocessing & annotation pipeline for `NOTEEVENTS` | §2 – §6 |
| 2. Supervised fine-tuning with LoRA | §8 |
| 3. Extraction of problems, medications, tests, results, negation, uncertainty, temporality into structured JSON | §4 (schema), §5 (LLM extractor) |
| 4. Comparison vs. pretrained model and conventional NLP baseline | §9 |
| 5. Accuracy, reliability, unsupported information, efficiency, generalisation to held-out patients | §10 – §11 |

### How the two models are used
* **Qwen3.6-35B-A3B (vLLM)** — (a) *pre-annotator*: produces **silver** labels that human annotators correct into **gold** labels; (b) a strong zero-shot comparator.
* **Qwen2.5-3B-Instruct** — the model that is **LoRA fine-tuned** and compared against its own un-tuned version.

> ⚠️ **Methodological note.** The test set must be scored against **human-verified gold annotations**, never against raw teacher output — otherwise the evaluation is circular. Silver labels are used only to bootstrap training data and to speed up annotation.

> 🔒 **Data governance.** MIMIC-III is credentialed data (PhysioNet DUA). Everything here runs locally on the DGX Spark; no note text leaves the machine (the vLLM endpoint is `localhost`). Do not commit outputs containing note text to public repositories.

## ▶ HOW TO RUN THIS NOTEBOOK (after a kernel restart)

Every code cell starts with a banner:

| Banner | Meaning |
|---|---|
| `✅ RUN · STEP n` | Run these **in step order, top to bottom** |
| `⏭️ SKIP` | Not needed: heavy step whose results are already saved on disk |
| `⛔ DO NOT RUN` | Would overwrite saved work |

**Do not use "Run All".** Click each ✅ cell and press Shift+Enter, skipping ⏭️ and ⛔ cells.

**Before step 17 (training):** in step 1 set `RUN_TRAINING = True`.

**Long steps (GPU):** step 17 (training, 1-3 h) and step 22 (only the FT model runs, ~1-2 h). Keep the temperature log running:
`nvidia-smi --query-gpu=timestamp,temperature.gpu,power.draw --format=csv -l 30 >> ~/gpu_log.csv &`

**Data safety:** outputs were cleared. Clear outputs again (Edit → Clear All Outputs) before sharing this notebook.

## 1. Environment setup

Install once (uncomment if needed). On DGX Spark (ARM64 / Grace-Blackwell) use the NVIDIA PyTorch container or the CUDA-enabled wheels that ship with DGX OS.

In [ ]:
# ════════ ⏭️ SKIP · installs (only once) ════════
# !pip install -q duckdb pandas numpy scikit-learn tqdm pydantic openai rapidfuzz matplotlib
# !pip install -q "transformers>=4.45" "peft>=0.13" "trl>=0.20" datasets accelerate

In [ ]:
# ════════ ✅ RUN · STEP 1 · imports + config  (set RUN_TRAINING = True here when you want to train) ════════
import os, re, json, time, random, hashlib, platform, warnings
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Literal

import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# ---------------- Configuration ----------------
DB_PATH = os.path.expanduser(os.environ.get("MIMIC_DB", "~/mimic_project/mimic3.duckdb"))
client  = OpenAI(base_url="http://localhost:8001/v1", api_key="EMPTY")
MODEL   = "nvidia/Qwen3.6-35B-A3B-NVFP4"          # teacher / zero-shot comparator (vLLM)
STUDENT_BASE = "Qwen/Qwen2.5-3B-Instruct"          # model to LoRA fine-tune

OUT_DIR = Path(os.path.expanduser("~/mimic_project/ie_pipeline"))
for sub in ["data", "silver", "gold", "models", "preds", "results", "figures"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Pipeline knobs
NOTE_CATEGORIES   = ["Discharge summary", "Physician ", "Nursing"]   # note: MIMIC 'Physician ' has a trailing space
MIN_CHARS, MAX_CHUNK_CHARS = 200, 3000
N_ANNOTATION_UNITS = 600          # total chunks sampled for annotation (train+val+test)
DOUBLE_ANNOT_FRAC  = 0.20         # fraction of gold units independently double-annotated
LLM_WORKERS        = 16           # concurrent requests to vLLM
RUN_TRAINING       = False        # set True to launch LoRA training (§8)

print("DuckDB:", DB_PATH, "| exists:", os.path.exists(DB_PATH))
print("Outputs:", OUT_DIR)

### 1.1 Connectivity checks
Verify the vLLM server is up and the model is loaded, and that DuckDB opens read-only.

In [ ]:
# ════════ ✅ RUN · STEP 2 · check vLLM + DuckDB ════════
served = [m.id for m in client.models.list().data]
print("vLLM models:", served)
assert MODEL in served, f"{MODEL} not served — check the vLLM launch command"

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
    max_tokens=10, temperature=0,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print("Smoke test:", r.choices[0].message.content)

con = duckdb.connect(DB_PATH, read_only=True)
tables = con.execute("SELECT table_schema, table_name FROM information_schema.tables").df()
tables.head(30)

In [ ]:
# ════════ ✅ RUN · STEP 3 · find NOTEEVENTS table (fast) ════════
# Resolve the NOTEEVENTS table regardless of case / schema naming
match = tables[tables.table_name.str.lower() == "noteevents"]
assert len(match), "NOTEEVENTS table not found in the DuckDB file"
NOTE_TBL = f'"{match.iloc[0].table_schema}"."{match.iloc[0].table_name}"'
cols = con.execute(f"DESCRIBE {NOTE_TBL}").df()
COL = {c.lower(): c for c in cols.column_name}            # lower-case -> actual name
print(NOTE_TBL); cols[["column_name", "column_type"]]

## 2. Exploratory analysis of `NOTEEVENTS`
Volume by category, error-flagged notes, and note length distribution — these inform category selection and chunking.

In [ ]:
# ════════ ⏭️ SKIP · exploration only ════════
q = f'''
SELECT {COL['category']} AS category,
       COUNT(*)                                   AS n_notes,
       COUNT(DISTINCT {COL['subject_id']})        AS n_patients,
       SUM(CASE WHEN {COL['iserror']} = 1 THEN 1 ELSE 0 END) AS n_error,
       CAST(MEDIAN(LENGTH({COL['text']})) AS INT) AS median_chars
FROM {NOTE_TBL}
GROUP BY 1 ORDER BY 2 DESC
'''
cat_stats = con.execute(q).df()
cat_stats

In [ ]:
# ════════ ⏭️ SKIP · exploration chart only ════════
fig, ax = plt.subplots(figsize=(8, 4))
cat_stats.set_index("category")["n_notes"].sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.set_xlabel("Number of notes"); ax.set_title("MIMIC-III NOTEEVENTS by category")
plt.tight_layout(); plt.savefig(OUT_DIR / "figures/note_categories.png", dpi=200); plt.show()

## 3. Preprocessing pipeline (Objective 1)

Steps:
1. **Category selection** — discharge summaries, physician and nursing notes.
2. **Remove error-flagged notes** (`ISERROR = 1`).
3. **Text normalisation** — collapse whitespace; keep de-identification placeholders (`[** ... **]`) but replace them with typed tokens (e.g. `[DATE]`, `[NAME]`) so the model is not confused by them.
4. **Duplicate detection** — exact duplicates via SHA-1 of normalised text; near-duplicates (copy-forward notes) via MinHash-style shingle Jaccard within a patient.
5. **Quality checks** — minimum length, proportion of alphabetic characters.
6. **Chunking** — long notes split on section headers / paragraph boundaries into ≤ `MAX_CHUNK_CHARS` units. The *chunk* is the unit of annotation and extraction.

In [ ]:
# ════════ ⏭️ SKIP · heavy: loads 424k notes. Results already saved ════════
cats_sql = ", ".join("'" + c.replace("'", "''") + "'" for c in NOTE_CATEGORIES)
q = f'''
SELECT {COL['row_id']} AS row_id, {COL['subject_id']} AS subject_id, {COL['hadm_id']} AS hadm_id,
       {COL['chartdate']} AS chartdate, {COL['category']} AS category,
       {COL['description']} AS description, {COL['text']} AS text
FROM {NOTE_TBL}
WHERE {COL['category']} IN ({cats_sql})
  AND ({COL['iserror']} IS NULL OR {COL['iserror']} <> 1)
'''
notes = con.execute(q).df()
notes["category"] = notes["category"].str.strip()
print(f"{len(notes):,} notes | {notes.subject_id.nunique():,} patients")
notes.category.value_counts()

In [ ]:
# ════════ ⏭️ SKIP · heavy: cleaning. Results already saved ════════
DEID_MAP = [
    (r"\[\*\*[^\]]*?(Date|Year|Month|Day)[^\]]*?\*\*\]", "[DATE]"),
    (r"\[\*\*\d{4}-\d{1,2}-\d{1,2}\*\*\]",                "[DATE]"),
    (r"\[\*\*\d{1,2}-\d{1,2}\*\*\]",                      "[DATE]"),
    (r"\[\*\*[^\]]*?(Name|Initials)[^\]]*?\*\*\]",        "[NAME]"),
    (r"\[\*\*[^\]]*?(Hospital|Location|Ward|Address|State|Country)[^\]]*?\*\*\]", "[LOCATION]"),
    (r"\[\*\*[^\]]*?(Telephone|Phone|Fax|Pager)[^\]]*?\*\*\]", "[PHONE]"),
    (r"\[\*\*[^\]]*?\*\*\]",                              "[REDACTED]"),
]
def normalise(text: str) -> str:
    t = text.replace("\r", "")
    for pat, rep in DEID_MAP:
        t = re.sub(pat, rep, t, flags=re.I)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

notes["clean"]  = notes["text"].map(normalise)
notes["n_char"] = notes["clean"].str.len()
notes["alpha_ratio"] = notes["clean"].map(lambda s: sum(c.isalpha() for c in s) / max(len(s), 1))

# exact duplicates
notes["sha1"] = notes["clean"].map(lambda s: hashlib.sha1(s.lower().encode()).hexdigest())
n0 = len(notes)
notes = notes.drop_duplicates("sha1")
print(f"Exact duplicates removed: {n0 - len(notes):,}")

# quality filter
n0 = len(notes)
notes = notes[(notes.n_char >= MIN_CHARS) & (notes.alpha_ratio >= 0.5)]
print(f"Low-quality notes removed: {n0 - len(notes):,}  |  remaining: {len(notes):,}")

### 3.1 Near-duplicate (copy-forward) detection
Nursing and physician progress notes are frequently copy-forwarded. We flag notes whose 5-word-shingle Jaccard similarity with an earlier note **from the same patient** exceeds 0.8, and drop them from the annotation pool (they would inflate apparent performance).

In [ ]:
# ════════ ⏭️ SKIP · very slow: near-duplicate search. Results already saved ════════
def shingles(s, k=5):
    w = re.findall(r"\w+", s.lower())
    return {hash(" ".join(w[i:i+k])) for i in range(max(len(w) - k + 1, 1))}

def flag_near_dups(df, thr=0.8, max_per_patient=200):
    drop = set()
    for sid, g in tqdm(df.sort_values("chartdate").groupby("subject_id"), desc="near-dup"):
        g = g.head(max_per_patient)
        seen = []
        for rid, txt in zip(g.row_id, g.clean):
            sh = shingles(txt)
            if any(len(sh & s) / max(len(sh | s), 1) >= thr for s in seen):
                drop.add(rid)
            else:
                seen.append(sh)
    return drop

# Run on a patient sample first if memory/time is limited; the full run is O(notes per patient^2)
near_dup_ids = flag_near_dups(notes)
notes = notes[~notes.row_id.isin(near_dup_ids)].reset_index(drop=True)
print(f"Near-duplicates removed: {len(near_dup_ids):,} | final notes: {len(notes):,}")

### 3.2 Section-aware chunking

In [ ]:
# ════════ ⏭️ SKIP · heavy: chunking. Results already saved ════════
SECTION_RE = re.compile(r"\n(?=[A-Z][A-Za-z /&]{2,40}:)")   # e.g. 'History of Present Illness:'

def _segments(text, max_chars):
    # sections -> paragraphs -> sentences -> hard windows, so no text is ever truncated
    for sec in (p.strip() for p in SECTION_RE.split(text)):
        if not sec: continue
        if len(sec) <= max_chars: yield sec; continue
        for para in (p.strip() for p in re.split(r"\n\s*\n", sec)):
            if not para: continue
            if len(para) <= max_chars: yield para; continue
            for sent in re.split(r"(?<=[.;])\s+|\n", para):
                for k in range(0, len(sent), max_chars):
                    if sent[k:k + max_chars].strip(): yield sent[k:k + max_chars].strip()

def chunk_note(text: str, max_chars=MAX_CHUNK_CHARS):
    chunks, buf = [], ""
    for seg in _segments(text, max_chars):           # greedy packing of segments
        if buf and len(buf) + len(seg) + 2 > max_chars:
            chunks.append(buf); buf = seg
        else:
            buf += ("\n\n" if buf else "") + seg
    if buf: chunks.append(buf)
    return [c for c in chunks if len(c) >= 100]

rows = []
for r in tqdm(notes.itertuples(), total=len(notes), desc="chunking"):
    for i, ch in enumerate(chunk_note(r.clean)):
        rows.append(dict(unit_id=f"{r.row_id}_{i}", row_id=r.row_id, subject_id=r.subject_id,
                         hadm_id=r.hadm_id, category=r.category, chunk_idx=i, text=ch))
units = pd.DataFrame(rows)
print(f"{len(units):,} chunks from {units.row_id.nunique():,} notes")
units.groupby("category").text.apply(lambda s: s.str.len().describe()).unstack().round(0)

### 3.3 Patient-level train / validation / test split
Splitting by `SUBJECT_ID` guarantees that no patient appears in more than one partition, preventing leakage of patient-specific language (Objective 5: generalisation to held-out patients).

In [ ]:
# ════════ ⏭️ SKIP · patient split. Results already saved ════════
from sklearn.model_selection import GroupShuffleSplit

patients = units.subject_id.unique()
rng = np.random.default_rng(SEED)
rng.shuffle(patients)
n = len(patients)
split_of = {}
for i, p in enumerate(patients):
    split_of[p] = "train" if i < 0.70 * n else ("val" if i < 0.85 * n else "test")
units["split"] = units.subject_id.map(split_of)

# leakage assertion
s = units.groupby("subject_id").split.nunique()
assert (s == 1).all(), "Patient appears in multiple splits!"
units.groupby("split").agg(chunks=("unit_id", "size"), notes=("row_id", "nunique"), patients=("subject_id", "nunique"))

### 3.4 Sample the annotation subset
Stratified by split × category, with **at most one chunk per note** and a cap per patient to maximise diversity.

In [ ]:
# ════════ ⛔ DO NOT RUN · would re-sample and OVERWRITE annotation_units.parquet ════════
SPLIT_FRAC = {"train": 0.70, "val": 0.10, "test": 0.20}

def sample_units(units, n_total):
    out = []
    for sp, frac in SPLIT_FRAC.items():
        pool = (units[units.split == sp]
                .sample(frac=1, random_state=SEED)
                .drop_duplicates("row_id")                          # 1 chunk per note
                .groupby("subject_id").head(2))                     # ≤2 notes per patient
        n_sp = int(n_total * frac)
        per_cat = n_sp // len(NOTE_CATEGORIES)
        for cat in pool.category.unique():
            out.append(pool[pool.category == cat].head(per_cat))
    return pd.concat(out, ignore_index=True)

ann = sample_units(units, N_ANNOTATION_UNITS)
ann.to_parquet(OUT_DIR / "data/annotation_units.parquet")
pd.crosstab(ann.split, ann.category, margins=True)

In [ ]:
# ════════ ✅ RUN · STEP 4 · RELOAD saved sample instead of steps above ════════
ann = pd.read_parquet(OUT_DIR / "data/annotation_units.parquet")
print(len(ann), "chunks loaded")
pd.crosstab(ann.split, ann.category, margins=True)


## 4. Extraction schema & annotation guideline (Objective 3)

Each chunk yields a JSON object with a list of **entities**. Every entity must carry a **verbatim evidence span** copied from the note, which is what allows us to measure *unsupported* (hallucinated) extractions.

| Field | Values | Notes |
|---|---|---|
| `text` | string | Canonical mention as written (e.g. "pneumonia", "metoprolol") |
| `type` | `PROBLEM` · `MEDICATION` · `TEST` · `RESULT` | Problems include diagnoses, symptoms, findings |
| `evidence` | string | Exact substring of the note supporting the entity |
| `negated` | bool | "denies chest pain", "no PE" |
| `certainty` | `certain` · `uncertain` · `hypothetical` | "possible PNA", "r/o MI" → uncertain; "if fever, give…" → hypothetical |
| `temporality` | `current` · `historical` · `future` | "h/o CABG" → historical; "will start heparin" → future |
| `related_test` | string \| null | For `RESULT` only — the test it belongs to (e.g. "WBC") |
| `value` | string \| null | For `RESULT` only — value + unit (e.g. "14.2 K/uL") |

In [ ]:
# ════════ ✅ RUN · STEP 5 · schema + guideline ════════
class Entity(BaseModel):
    text: str
    type: Literal["PROBLEM", "MEDICATION", "TEST", "RESULT"]
    evidence: str
    negated: bool = False
    certainty: Literal["certain", "uncertain", "hypothetical"] = "certain"
    temporality: Literal["current", "historical", "future"] = "current"
    related_test: Optional[str] = None
    value: Optional[str] = None

class Extraction(BaseModel):
    entities: List[Entity] = Field(default_factory=list)

JSON_SCHEMA = Extraction.model_json_schema()

GUIDELINE = '''You are a clinical information extraction system. Extract every clinically relevant entity from the ICU note excerpt.

Entity types:
- PROBLEM: diagnoses, diseases, symptoms, signs, abnormal findings, complications.
- MEDICATION: drugs, IV fluids, blood products, drug classes when given/considered.
- TEST: laboratory tests, imaging, procedures performed for diagnosis, vital-sign measurements.
- RESULT: a reported test result. Put the test name in related_test and the value with units in value.

Attributes:
- negated: true if the entity is explicitly absent/denied/ruled out ("no", "denies", "negative for", "without").
- certainty: "uncertain" for possible/probable/suspected/likely/?/r/o/concern for; "hypothetical" for conditional or instructional mentions ("if", "should", "return if"); else "certain".
- temporality: "historical" for past history / prior episodes (h/o, s/p, PMH section); "future" for planned items; else "current".

Rules:
1. "evidence" MUST be copied verbatim from the note (shortest phrase that supports the entity, ≤ 25 words). Never paraphrase.
2. Do not infer entities that are not stated. If nothing is present return {"entities": []}.
3. Ignore de-identification tokens such as [DATE], [NAME].
4. Expand nothing; keep abbreviations as written in "text".
Return ONLY valid JSON matching the schema.'''

def build_messages(note_text: str):
    return [{"role": "system", "content": GUIDELINE},
            {"role": "user", "content": f"NOTE EXCERPT:\n<<<\n{note_text}\n>>>\n\nReturn the JSON."}]

print(json.dumps(JSON_SCHEMA, indent=1)[:800], "...")

## 5. LLM extractor via vLLM (teacher / zero-shot)

* **Structured decoding**: vLLM's `response_format={"type": "json_schema"}` constrains generation to the schema (guaranteed parseable output). We also keep an *unconstrained* mode because JSON-validity is itself a reported metric (§10).
* **Thinking disabled** for Qwen3.x (`enable_thinking: False`) to keep outputs short and deterministic.
* Latency and token usage are logged per call.

In [ ]:
# ════════ ✅ RUN · STEP 6 · base functions ════════
def parse_json(raw: str):
    # Return (Extraction | None, is_valid_json, is_schema_valid)
    if raw is None: return None, False, False
    s = raw.strip()
    s = re.sub(r"^```(?:json)?|```$", "", s, flags=re.M).strip()
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.S).strip()
    try:
        obj = json.loads(s)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", s, flags=re.S)
        if not m: return None, False, False
        try: obj = json.loads(m.group(0))
        except json.JSONDecodeError: return None, False, False
    try:
        return Extraction.model_validate(obj), True, True
    except ValidationError:
        # salvage valid entities
        ents = []
        for e in (obj.get("entities", []) if isinstance(obj, dict) else []):
            try: ents.append(Entity.model_validate(e))
            except ValidationError: pass
        return Extraction(entities=ents), True, False


def llm_extract(text: str, model=MODEL, constrained=True, max_tokens=3000, retries=2):
    kwargs = dict(model=model, messages=build_messages(text), temperature=0.0,
                  max_tokens=max_tokens, seed=SEED,
                  extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    if constrained:
        kwargs["response_format"] = {"type": "json_schema",
                                     "json_schema": {"name": "extraction", "schema": JSON_SCHEMA}}
    for attempt in range(retries + 1):
        try:
            t0 = time.perf_counter()
            r = client.chat.completions.create(**kwargs)
            lat = time.perf_counter() - t0
            raw = r.choices[0].message.content
            ext, ok_json, ok_schema = parse_json(raw)
            return dict(raw=raw, extraction=ext.model_dump() if ext else None,
                        valid_json=ok_json, valid_schema=ok_schema, latency_s=lat,
                        prompt_tokens=r.usage.prompt_tokens, completion_tokens=r.usage.completion_tokens,
                        finish_reason=r.choices[0].finish_reason)
        except Exception as e:
            if attempt == retries:
                return dict(raw=None, extraction=None, valid_json=False, valid_schema=False,
                            latency_s=None, error=str(e))
            time.sleep(2 ** attempt)


def run_batch(df, out_path, workers=LLM_WORKERS, **kw):
    # Run extraction over a DataFrame of units with caching (resumable)
    done = {}
    if Path(out_path).exists():
        for line in open(out_path):
            d = json.loads(line); done[d["unit_id"]] = d
    todo = df[~df.unit_id.isin(done)]
    with open(out_path, "a") as f, ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(llm_extract, t, **kw): u for u, t in zip(todo.unit_id, todo.text)}
        for fut in tqdm(as_completed(futs), total=len(futs), desc=Path(out_path).stem):
            d = fut.result(); d["unit_id"] = futs[fut]
            f.write(json.dumps(d) + "\n"); done[d["unit_id"]] = d
    return pd.DataFrame([done[u] for u in df.unit_id if u in done])

In [ ]:
# ════════ ✅ RUN · STEP 7 · improved extractor (demo switched off) ════════
# ============================================================
# CELL 15 (UPDATED): improved extractor + demo on one chunk
# ============================================================

# --- 1. Better instructions (shorter evidence, no repeats, compact JSON) ---
if "Output compact single-line JSON" not in GUIDELINE:          # avoid adding twice if re-run
    GUIDELINE = GUIDELINE.replace("≤ 25 words", "≤ 12 words")
    GUIDELINE += (
        "\n5. List each distinct entity only once. Do not repeat entries."
        "\n6. Output compact single-line JSON with no indentation or line breaks. "
        "Omit related_test and value unless type is RESULT."
    )

# --- 2. One call to the model ---
def _call(text, model, constrained, max_tokens):
    kwargs = dict(
        model=model, messages=build_messages(text),
        temperature=0.0, max_tokens=max_tokens, seed=SEED,
        extra_body={"chat_template_kwargs": {"enable_thinking": False},
                    "repetition_penalty": 1.05},
    )
    if constrained:
        kwargs["response_format"] = {"type": "json_schema",
                                     "json_schema": {"name": "extraction", "schema": JSON_SCHEMA}}
    t0 = time.perf_counter()
    r = client.chat.completions.create(**kwargs)
    return r, time.perf_counter() - t0

# --- 3. Cut text roughly in half at a line break ---
def _split_half(text):
    mid = len(text) // 2
    cut = text.rfind("\n", 0, mid)
    if cut < len(text) * 0.25:
        cut = mid
    return text[:cut], text[cut:]

# --- 4. Main extractor: if output is cut off, split the note and merge ---
def llm_extract(text, model=MODEL, constrained=True, max_tokens=8192, retries=2, depth=0):
    for attempt in range(retries + 1):
        try:
            r, lat = _call(text, model, constrained, max_tokens)
            raw = r.choices[0].message.content
            fin = r.choices[0].finish_reason

            # Output too long -> split into two halves (max 2 levels deep)
            if fin == "length" and depth < 2 and len(text) > 400:
                a, b = _split_half(text)
                ra = llm_extract(a, model, constrained, max_tokens, retries, depth + 1)
                rb = llm_extract(b, model, constrained, max_tokens, retries, depth + 1)

                merged, seen = [], set()
                for e in (ra["extraction"] or {"entities": []})["entities"] + \
                         (rb["extraction"] or {"entities": []})["entities"]:
                    key = (e["type"], e["text"].lower().strip())
                    if key not in seen:
                        seen.add(key); merged.append(e)

                return dict(
                    raw=None, extraction={"entities": merged},
                    valid_json=ra["valid_json"] and rb["valid_json"],
                    valid_schema=ra["valid_schema"] and rb["valid_schema"],
                    latency_s=lat + (ra["latency_s"] or 0) + (rb["latency_s"] or 0),
                    prompt_tokens=None,
                    completion_tokens=(ra["completion_tokens"] or 0) + (rb["completion_tokens"] or 0),
                    finish_reason="split",
                    n_splits=1 + ra["n_splits"] + rb["n_splits"],
                )

            # Normal case
            ext, ok_json, ok_schema = parse_json(raw)
            return dict(
                raw=raw, extraction=ext.model_dump() if ext else None,
                valid_json=ok_json, valid_schema=ok_schema, latency_s=lat,
                prompt_tokens=r.usage.prompt_tokens,
                completion_tokens=r.usage.completion_tokens,
                finish_reason=fin, n_splits=0,
            )

        except Exception as e:
            if attempt == retries:
                return dict(raw=None, extraction=None, valid_json=False, valid_schema=False,
                            latency_s=None, prompt_tokens=None, completion_tokens=None,
                            finish_reason="error", n_splits=0, error=str(e))
            time.sleep(2 ** attempt)

# --- 5. Demo on one chunk (off by default: calls the model, prints note text) ---
RUN_DEMO = False
if RUN_DEMO:
    demo = ann.iloc[0]
    print(demo.category, "|", demo.unit_id)
    print(demo.text[:800], "\n" + "-" * 80)

    res = llm_extract(demo.text)
    print(f"latency={res['latency_s']:.1f}s | finish={res['finish_reason']} | "
          f"splits={res['n_splits']} | valid_schema={res['valid_schema']} | "
          f"tokens_out={res['completion_tokens']}")

    if res["extraction"] and res["extraction"]["entities"]:
        df_demo = pd.DataFrame(res["extraction"]["entities"])
        print(f"\n{len(df_demo)} entities found")
        print(df_demo.type.value_counts().to_string())
        display(df_demo)
    else:
        print("No extraction. Details:\n", (res["raw"] or res.get("error", ""))[-800:])


### 5.1 Evidence verification — detecting unsupported extractions
An entity is **supported** if its `evidence` string occurs in the source text (after whitespace/case normalisation), falling back to a fuzzy partial match ≥ 90 (RapidFuzz) to tolerate minor tokenisation differences. It is additionally checked that the entity `text` appears within (or fuzzily within) the evidence. Anything else counts toward the **unsupported extraction rate**.

In [ ]:
# ════════ ✅ RUN · STEP 8 · rapidfuzz + support check (demo switched off) ════════
# Install rapidfuzz once (safe to leave in; skips if already installed)
%pip install -q rapidfuzz

from rapidfuzz import fuzz

def _norm(s):
    return re.sub(r"\s+", " ", (s or "").lower()).strip()

def is_supported(ent: dict, source: str, thr=90) -> bool:
    # True if the evidence quote really exists in the note
    src = _norm(source)
    ev  = _norm(ent.get("evidence"))
    tx  = _norm(ent.get("text"))
    if not ev or not tx:
        return False
    ev_ok = ev in src or fuzz.partial_ratio(ev, src) >= thr
    tx_ok = tx in ev or tx in src or fuzz.partial_ratio(tx, ev) >= thr
    return ev_ok and tx_ok

def add_support_flags(extraction: dict, source: str):
    if not extraction:
        return extraction
    for e in extraction["entities"]:
        e["supported"] = is_supported(e, source)
    return extraction

# Check the demo note (only if the demo ran above)
if RUN_DEMO and "res" in dir():
    add_support_flags(res["extraction"], demo.text)
    df_check = pd.DataFrame(res["extraction"]["entities"])
    print(f"Supported: {df_check.supported.sum()} / {len(df_check)} "
          f"({df_check.supported.mean():.0%})")
    df_check[["text", "type", "supported", "evidence"]]


In [ ]:
# ════════ ✅ RUN · STEP 9 · FIX A  (always run right after step 8) ════════
# ===== FIX A: forgiving parser + "..." support check =====
TYPE_MAP = {"problem":"PROBLEM","diagnosis":"PROBLEM","symptom":"PROBLEM","condition":"PROBLEM",
            "finding":"PROBLEM","disease":"PROBLEM","medication":"MEDICATION","drug":"MEDICATION",
            "med":"MEDICATION","medicine":"MEDICATION","treatment":"MEDICATION","test":"TEST",
            "lab":"TEST","procedure":"TEST","imaging":"TEST","investigation":"TEST",
            "result":"RESULT","value":"RESULT","lab_result":"RESULT","test_result":"RESULT"}
CERT_MAP = {"certain":"certain","definite":"certain","uncertain":"uncertain","possible":"uncertain",
            "probable":"uncertain","hypothetical":"hypothetical","conditional":"hypothetical"}
TEMP_MAP = {"current":"current","present":"current","historical":"historical","past":"historical",
            "history":"historical","future":"future","planned":"future"}

def _fix_entity(e):
    # Map near-miss keys and values onto the schema
    if not isinstance(e, dict): return None
    e = {str(k).lower(): v for k, v in e.items()}
    text = e.get("text") or e.get("entity") or e.get("name") or e.get("mention")
    typ  = TYPE_MAP.get(str(e.get("type") or e.get("category") or e.get("label") or "").strip().lower())
    if not text or not typ:                       # e.g. {"PROBLEM": "pneumonia", ...}
        for k, v in e.items():
            if k in ("problem", "medication", "test", "result") and isinstance(v, str) and v.strip():
                text, typ = v, k.upper(); break
    if not text or not typ: return None
    neg = e.get("negated", e.get("negation", False))
    if isinstance(neg, str): neg = neg.strip().lower() in ("true", "yes", "negated", "absent", "1")
    return dict(text=str(text), type=typ,
                evidence=str(e.get("evidence") or e.get("span") or text),
                negated=bool(neg),
                certainty=CERT_MAP.get(str(e.get("certainty", "certain")).lower(), "certain"),
                temporality=TEMP_MAP.get(str(e.get("temporality", "current")).lower(), "current"),
                related_test=e.get("related_test"), value=None if e.get("value") is None else str(e.get("value")))

def parse_json(raw: str):
    # Returns (Extraction|None, valid_json, schema_valid). schema_valid stays STRICT.
    if raw is None: return None, False, False
    s = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)
    s = re.sub(r"^```(?:json)?|```$", "", s.strip(), flags=re.M).strip()
    try:
        obj = json.loads(s)
    except json.JSONDecodeError:
        m = re.search(r"[\{\[].*[\}\]]", s, flags=re.S)
        try: obj = json.loads(m.group(0)) if m else None
        except json.JSONDecodeError: obj = None
        if obj is None:                                   # invalid / cut-off JSON: recover complete items
            items, i = [], s.find("[")
            pos = i + 1 if i >= 0 else len(s)
            while True:
                j = s.find("{", pos)
                if j < 0: break
                try: o, pos = json.JSONDecoder().raw_decode(s, j); items.append(o)
                except json.JSONDecodeError: break
            ents = []
            for e in items:
                fe = _fix_entity(e)
                if fe:
                    try: ents.append(Entity.model_validate(fe))
                    except Exception: pass
            return Extraction(entities=ents), False, False
    if isinstance(obj, dict) and isinstance(obj.get("entities"), list):   # strict = exact schema
        try:
            return Extraction.model_validate(obj), True, True
        except Exception:
            pass
    items = obj if isinstance(obj, list) else next(
        (v for k, v in obj.items() if isinstance(v, list)), []) if isinstance(obj, dict) else []
    ents = []
    for e in items:
        fe = _fix_entity(e)
        if fe:
            try: ents.append(Entity.model_validate(fe))
            except Exception: pass
    return Extraction(entities=ents), True, False

def is_supported(ent: dict, source: str, thr=90) -> bool:
    # Supports evidence with '...' gaps: every fragment must be found in the note
    src, tx = _norm(source), _norm(ent.get("text"))
    frags = [_norm(f) for f in re.split(r"\.{2,}|…", ent.get("evidence") or "")]
    frags = [f for f in frags if len(f) >= 3]
    if not frags or not tx: return False
    ev_ok = all(f in src or fuzz.partial_ratio(f, src) >= thr for f in frags)
    tx_ok = tx in src or any(tx in f for f in frags) or fuzz.partial_ratio(tx, src) >= thr
    return ev_ok and tx_ok

if "Never use '...'" not in GUIDELINE:
    GUIDELINE += "\n7. Never use '...' or skip words inside evidence; copy one continuous phrase."
print("✅ Fix A loaded")

## 6. Annotation workflow: silver → gold

1. **Silver pre-annotation** of all sampled units with the teacher model (constrained decoding).
2. Unsupported entities are automatically dropped from silver labels.
3. Export to an annotator-friendly JSONL (compatible with Label Studio / Doccano-style review, or plain editing).
4. Annotators correct according to the guideline (§4). A `DOUBLE_ANNOT_FRAC` subset of **val+test** units is annotated **independently by two annotators** *without* the silver pre-fill to avoid anchoring bias, and agreement is computed (§6.2).
5. Corrected files are saved to `gold/` and loaded back.

In [ ]:
# ════════ ⛔ DO NOT RUN · disabled: would delete progress of the silver run ════════
# This cell MOVED the silver cache away. Running it again would force the
# whole 2.5-hour Qwen3.6 run to start over. It is disabled on purpose.

# cache = OUT_DIR / "silver/teacher_constrained.jsonl"
# if cache.exists():
#     cache.rename(cache.with_suffix(".old.jsonl"))   # kept as a backup, not deleted
#     print("Old cache moved to:", cache.with_suffix(".old.jsonl").name)


In [ ]:
# ════════ ✅ RUN · STEP 10 · safe run_batch ════════
import threading

def run_batch(df, out_path, workers=LLM_WORKERS, **kw):
    # Run extraction on many chunks in parallel, with a resumable cache.
    # - skips broken/half-written lines instead of crashing
    # - re-tries chunks that previously failed
    # - writes each line safely (one at a time) and flushes immediately
    out_path = Path(out_path)
    done, bad = {}, 0

    if out_path.exists():
        for line in open(out_path):
            try:
                d = json.loads(line)
            except json.JSONDecodeError:
                bad += 1
                continue
            if d.get("extraction") is not None:          # keep only successful results
                done[d["unit_id"]] = d
        if bad:
            print(f"Skipped {bad} broken line(s) in cache")
        print(f"Resuming: {len(done)} already done")

    todo = df[~df.unit_id.isin(done)]
    lock = threading.Lock()

    with open(out_path, "a") as f, ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(llm_extract, t, **kw): u for u, t in zip(todo.unit_id, todo.text)}
        for fut in tqdm(as_completed(futs), total=len(futs), desc=out_path.stem):
            d = fut.result()
            d["unit_id"] = futs[fut]
            with lock:
                f.write(json.dumps(d) + "\n")
                f.flush()                                  # save to disk right away
            done[d["unit_id"]] = d

    out = pd.DataFrame([done[u] for u in df.unit_id if u in done])
    print(f"Finished: {len(out)} chunks | failed: {out.extraction.isna().sum()} "
          f"| split: {(out.finish_reason == 'split').sum()}")
    return out

In [ ]:
# ════════ ✅ RUN · STEP 11 · re-clean silver labels from cache (seconds, no model calls) ════════
silver_raw = run_batch(ann, OUT_DIR / "silver/teacher_constrained.jsonl", constrained=True)
silver_raw = silver_raw.merge(ann[["unit_id", "text"]], on="unit_id")

def clean_silver(row):
    ext = add_support_flags(row.extraction, row.text) if row.extraction else {"entities": []}
    ents = [{k: v for k, v in e.items() if k != "supported"} for e in ext["entities"] if e.get("supported")]
    return {"entities": ents}

silver_raw["silver"] = silver_raw.apply(clean_silver, axis=1)
print("Schema-valid:", silver_raw.valid_schema.mean().round(3),
      "| mean latency (s):", silver_raw.latency_s.mean().round(2),
      "| mean entities/unit:", silver_raw.silver.map(lambda x: len(x["entities"])).mean().round(1))

silver_raw.to_parquet(OUT_DIR / "silver/silver_clean.parquet")
print("✅ Saved silver_clean.parquet")


In [ ]:
# ════════ ✅ RUN · STEP 12 · re-export annotator files with corrected labels ════════
# Export for annotation
rng = np.random.default_rng(SEED)
eval_units = ann[ann.split.isin(["val", "test"])].unit_id.tolist()
double_ids = set(rng.choice(eval_units, size=int(len(eval_units) * DOUBLE_ANNOT_FRAC), replace=False))

def export_for_annotation(path, prefill=True, only=None):
    with open(path, "w") as f:
        for r in silver_raw.merge(ann[["unit_id", "split", "category"]], on="unit_id").itertuples():
            if only is not None and r.unit_id not in only: continue
            f.write(json.dumps({"unit_id": r.unit_id, "split": r.split, "category": r.category,
                                "text": r.text,
                                "entities": r.silver["entities"] if prefill else []}) + "\n")

export_for_annotation(OUT_DIR / "gold/to_annotate_prefilled.jsonl", prefill=True)
export_for_annotation(OUT_DIR / "gold/double_annot_A_blank.jsonl", prefill=False, only=double_ids)
export_for_annotation(OUT_DIR / "gold/double_annot_B_blank.jsonl", prefill=False, only=double_ids)
print(f"Exported {len(silver_raw)} units for annotation; {len(double_ids)} for independent double annotation.")

### 6.1 Load gold annotations
Expected file: `gold/gold_annotations.jsonl` — one line per unit `{"unit_id", "entities": [...]}`.
Until human annotation is complete, the cell falls back to silver labels for **train/val only** so the rest of the notebook can be developed end-to-end. The **test** set is always gold-only.

In [ ]:
# ════════ ✅ RUN · STEP 13 · load gold/silver labels ════════
GOLD_PATH = OUT_DIR / "gold/gold_annotations.jsonl"

def load_jsonl_labels(path):
    out = {}
    for line in open(path):
        d = json.loads(line)
        out[d["unit_id"]] = {"entities": [Entity.model_validate(e).model_dump() for e in d["entities"]]}
    return out

gold = load_jsonl_labels(GOLD_PATH) if GOLD_PATH.exists() else {}
silver = dict(zip(silver_raw.unit_id, silver_raw.silver))

labels = {}
for r in ann.itertuples():
    if r.unit_id in gold:
        labels[r.unit_id] = (gold[r.unit_id], "gold")
    elif r.split in ("train", "val"):
        labels[r.unit_id] = (silver[r.unit_id], "silver")
ann["label_source"] = ann.unit_id.map(lambda u: labels.get(u, (None, "missing"))[1])
pd.crosstab(ann.split, ann.label_source)

### 6.2 Inter-annotator agreement
For span-extraction tasks, Cohen's κ on tokens is ill-defined, so agreement is reported as **pairwise entity-level F1** (treating annotator A as reference) plus **Cohen's κ for each attribute** on entities both annotators identified.

In [ ]:
# ════════ ✅ RUN · STEP 14 · inter-annotator agreement (works once A/B files exist) ════════
from sklearn.metrics import cohen_kappa_score

def norm_text(s): return re.sub(r"[^a-z0-9 ]", "", re.sub(r"\s+", " ", (s or "").lower())).strip()

def match_entities(gold_ents, pred_ents, mode="relaxed", thr=0.5):
    # Greedy one-to-one matching within the same type. Returns (pairs, unmatched_gold, unmatched_pred).
    def toks(s): return set(norm_text(s).split())
    pairs, used = [], set()
    for gi, g in enumerate(gold_ents):
        best, best_j = None, -1
        for pj, p in enumerate(pred_ents):
            if pj in used or p["type"] != g["type"]: continue
            if mode == "strict":
                sc = 1.0 if norm_text(p["text"]) == norm_text(g["text"]) else 0.0
            else:
                a, b = toks(g["text"]), toks(p["text"])
                sc = len(a & b) / max(len(a | b), 1)
            if sc >= (1.0 if mode == "strict" else thr) and (best is None or sc > best):
                best, best_j = sc, pj
        if best_j >= 0:
            used.add(best_j); pairs.append((gi, best_j))
    mg = [i for i in range(len(gold_ents)) if i not in {a for a, _ in pairs}]
    mp = [j for j in range(len(pred_ents)) if j not in used]
    return pairs, mg, mp

def iaa(path_a, path_b):
    A, B = load_jsonl_labels(path_a), load_jsonl_labels(path_b)
    tp = fp = fn = 0; attr = defaultdict(lambda: ([], []))
    for u in set(A) & set(B):
        ga, gb = A[u]["entities"], B[u]["entities"]
        pairs, mg, mp = match_entities(ga, gb)
        tp += len(pairs); fn += len(mg); fp += len(mp)
        for i, j in pairs:
            for k in ["negated", "certainty", "temporality"]:
                attr[k][0].append(str(ga[i][k])); attr[k][1].append(str(gb[j][k]))
    f1 = 2 * tp / max(2 * tp + fp + fn, 1)
    out = {"n_units": len(set(A) & set(B)), "entity_F1": round(f1, 3)}
    for k, (a, b) in attr.items():
        out[f"kappa_{k}"] = round(cohen_kappa_score(a, b), 3) if len(set(a + b)) > 1 else float("nan")
    return out

pa, pb = OUT_DIR / "gold/double_annot_A.jsonl", OUT_DIR / "gold/double_annot_B.jsonl"
if pa.exists() and pb.exists():
    display(pd.Series(iaa(pa, pb)))
else:
    print("Double-annotation files not yet available:", pa.name, pb.name)

## 7. Build the supervised fine-tuning dataset

Conversational **prompt–completion** format: TRL computes loss only on the assistant completion (the JSON), not on the note text. The *same* system prompt and user template are used at training and inference time.

In [ ]:
# ════════ ✅ RUN · STEP 15 · build training dataset ════════
from datasets import Dataset

def to_sft(split):
    recs = []
    for r in ann[ann.split == split].itertuples():
        lab = labels.get(r.unit_id)
        if lab is None: continue
        recs.append({
            "unit_id": r.unit_id,
            "prompt": build_messages(r.text),
            "completion": [{"role": "assistant", "content": json.dumps(lab[0], ensure_ascii=False)}],
        })
    return Dataset.from_list(recs)

train_ds, val_ds = to_sft("train"), to_sft("val")
print(train_ds, val_ds)
train_ds.to_json(OUT_DIR / "data/sft_train.jsonl"); val_ds.to_json(OUT_DIR / "data/sft_val.jsonl")

## 8. LoRA fine-tuning of Qwen2.5-3B-Instruct (Objective 2)

| Hyper-parameter | Value |
|---|---|
| LoRA rank *r* / α / dropout | 16 / 32 / 0.05 |
| Target modules | all attention + MLP projections |
| Precision | bf16 |
| LR / schedule | 2e-4, cosine, 3 % warm-up |
| Epochs | 3 (early stopping on val loss) |
| Effective batch | 16 (4 × grad-accum 4) |
| Max sequence length | 4096 tokens |

> **DGX Spark memory note:** the GB10 has 128 GB unified memory shared between CPU and GPU. If the vLLM server is running concurrently, start it with a lower `--gpu-memory-utilization` (e.g. `0.45`) so training has headroom, or stop it during training.

In [ ]:
# ════════ ✅ RUN · STEP 16 · torch + LoRA imports ════════
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, EarlyStoppingCallback
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR = OUT_DIR / "models/qwen25_3b_lora_ie"
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# ════════ ✅ RUN · STEP 17 · LoRA TRAINING (1-3 h GPU) - needs RUN_TRAINING = True in step 1 ════════
if RUN_TRAINING:
    tok = AutoTokenizer.from_pretrained(STUDENT_BASE)
    model = AutoModelForCausalLM.from_pretrained(STUDENT_BASE, torch_dtype=torch.bfloat16,
                                                 attn_implementation="sdpa", device_map="auto")
    lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                          "gate_proj", "up_proj", "down_proj"])
    cfg = SFTConfig(
        output_dir=str(ADAPTER_DIR), num_train_epochs=3,
        per_device_train_batch_size=4, per_device_eval_batch_size=4, gradient_accumulation_steps=4,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03, weight_decay=0.0,
        bf16=True, gradient_checkpointing=True, max_length=4096,
        logging_steps=10, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        report_to="none", seed=SEED,
    )
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=val_ds,
                         processing_class=tok, peft_config=lora_cfg,
                         callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
    trainer.model.print_trainable_parameters()

    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    train_out = trainer.train()
    train_stats = {"train_time_min": (time.perf_counter() - t0) / 60,
                   "peak_gpu_mem_GB": torch.cuda.max_memory_allocated() / 1e9,
                   **train_out.metrics}
    trainer.save_model(str(ADAPTER_DIR)); tok.save_pretrained(str(ADAPTER_DIR))
    json.dump(train_stats, open(OUT_DIR / "results/train_stats.json", "w"), indent=2)
    print(train_stats)
else:
    print("RUN_TRAINING=False — skipping. Adapter expected at", ADAPTER_DIR)

In [ ]:
# ════════ ✅ RUN · STEP 18 · loss plot ════════
if RUN_TRAINING:
    hist = pd.DataFrame(trainer.state.log_history)
    fig, ax = plt.subplots(figsize=(7, 4))
    hist.dropna(subset=["loss"]).plot(x="step", y="loss", ax=ax, label="train")
    if "eval_loss" in hist: hist.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, marker="o", label="val")
    ax.set_title("LoRA SFT loss"); plt.tight_layout(); plt.savefig(OUT_DIR / "figures/loss.png", dpi=200); plt.show()

### 8.1 Serving the adapter (optional, faster evaluation)
The adapter can be served by vLLM alongside the base model, so all systems are evaluated through the same OpenAI-compatible interface:

```bash
vllm serve Qwen/Qwen2.5-3B-Instruct --port 8002 --max-model-len 8192 \
     --enable-lora --max-lora-rank 16 \
     --lora-modules ie-lora=~/mimic_project/ie_pipeline/models/qwen25_3b_lora_ie \
     --gpu-memory-utilization 0.25
```
Below, a Hugging Face `generate` path is used instead, which additionally lets us measure per-note latency and peak GPU memory directly.

In [ ]:
# ════════ ✅ RUN · STEP 19 · HF extractor ════════
class HFExtractor:
    # Local HF inference for the base student and the LoRA-adapted student.
    def __init__(self, base=STUDENT_BASE, adapter=None, merge=True):
        self.tok = AutoTokenizer.from_pretrained(base)
        m = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16, device_map="auto")
        if adapter:
            m = PeftModel.from_pretrained(m, str(adapter))
            if merge:
                m = m.merge_and_unload()
        self.model = m.eval()

    @torch.inference_mode()
    def __call__(self, text, max_new_tokens=4096):
        # Works with both old and new transformers versions
        enc = self.tok.apply_chat_template(
            build_messages(text), add_generation_prompt=True,
            return_tensors="pt", return_dict=True,
        ).to(self.model.device)
        n_in = enc["input_ids"].shape[1]

        torch.cuda.synchronize(); t0 = time.perf_counter()
        out = self.model.generate(
            **enc,                                   # input_ids + attention_mask
            max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=self.tok.eos_token_id,
        )
        torch.cuda.synchronize(); lat = time.perf_counter() - t0

        new_tokens = out[0, n_in:]
        raw = self.tok.decode(new_tokens, skip_special_tokens=True)
        ext, ok_json, ok_schema = parse_json(raw)
        return dict(
            raw=raw, extraction=ext.model_dump() if ext else None,
            valid_json=ok_json, valid_schema=ok_schema, latency_s=lat,
            completion_tokens=int(len(new_tokens)),
            finish_reason="length" if len(new_tokens) >= max_new_tokens else "stop",
        )


def run_hf(extractor, df, out_path):
    out_path = Path(out_path)
    done = {}
    if out_path.exists():
        for line in open(out_path):
            try:
                d = json.loads(line)
                done[d["unit_id"]] = d
            except json.JSONDecodeError:
                pass                                  # skip broken lines
    todo = df[~df.unit_id.isin(done)]
    torch.cuda.reset_peak_memory_stats()
    with open(out_path, "a") as f:
        for r in tqdm(todo.itertuples(), total=len(todo), desc=out_path.stem):
            d = extractor(r.text); d["unit_id"] = r.unit_id
            f.write(json.dumps(d) + "\n"); f.flush()
            done[r.unit_id] = d
    peak = torch.cuda.max_memory_allocated() / 1e9
    return pd.DataFrame([done[u] for u in df.unit_id if u in done]), peak

## 9. Systems compared (Objective 4)

| ID | System | Role |
|---|---|---|
| **B0** | Rule-based: lexicon + NegEx-style triggers | Conventional NLP baseline |
| **B1** | Qwen2.5-3B-Instruct, zero-shot (unconstrained) | Original pretrained model |
| **B2** | Qwen3.6-35B-A3B, zero-shot (vLLM, constrained) | Large-model reference / teacher |
| **FT** | Qwen2.5-3B-Instruct + LoRA | Proposed model |

All systems are run on the **gold test set** only.

In [ ]:
# ════════ ✅ RUN · STEP 20 · select test units ════════
test_units = ann[(ann.split == "test") & (ann.unit_id.isin(gold))].copy()
if len(test_units) == 0:
    print("⚠️ No gold test annotations yet — using silver test labels ONLY for pipeline debugging. "
          "Do NOT report these numbers.")
    test_units = ann[ann.split == "test"].copy()
    REF = {u: silver[u] for u in test_units.unit_id}
else:
    REF = {u: gold[u] for u in test_units.unit_id}
print(len(test_units), "test units")

### 9.1 B0 — Rule-based baseline
A dictionary is induced from the **training** labels (most frequent type per surface form, frequency ≥ 2). Mentions are matched with word boundaries; attributes are assigned with NegEx/ConText-style trigger windows within the same sentence.

In [ ]:
# ════════ ✅ RUN · STEP 21 · B0 rules baseline ════════
NEG_TRIG  = r"\b(no|not|denies|denied|without|negative for|absence of|free of|ruled out|r/o'd|resolved)\b"
UNC_TRIG  = r"\b(possible|possibly|probable|likely|suspected|suspect|concern for|cannot rule out|r/o|rule out|questionable|\?|may have|consistent with)\b"
HYP_TRIG  = r"\b(if|should|return if|in case|prn)\b"
HIST_TRIG = r"\b(h/o|history of|hx of|s/p|status post|prior|previous|past|pmh)\b"
FUT_TRIG  = r"\b(will|plan to|planned|to start|to be|consider|schedule[d]?)\b"

lex_counter = defaultdict(Counter)
for r in ann[ann.split == "train"].itertuples():
    for e in labels.get(r.unit_id, ({"entities": []},))[0]["entities"]:
        t = norm_text(e["text"])
        if 2 < len(t) < 60: lex_counter[t][e["type"]] += 1
LEXICON = {t: c.most_common(1)[0][0] for t, c in lex_counter.items() if sum(c.values()) >= 2}
LEX_RE = re.compile(r"\b(" + "|".join(sorted(map(re.escape, LEXICON), key=len, reverse=True)) + r")\b", re.I) if LEXICON else None
print(f"Lexicon size: {len(LEXICON):,}")

def rule_extract(text):
    t0 = time.perf_counter(); ents = []; seen = set()
    for sent in re.split(r"(?<=[.;\n])\s+", text):
        if not LEX_RE: break
        for m in LEX_RE.finditer(sent):
            key = norm_text(m.group(0))
            typ = LEXICON.get(key)
            if not typ or key in seen: continue
            seen.add(key)
            pre = sent[max(0, m.start() - 60):m.start()].lower()
            certainty = ("hypothetical" if re.search(HYP_TRIG, pre) else
                         "uncertain" if re.search(UNC_TRIG, pre) else "certain")
            temporality = ("historical" if re.search(HIST_TRIG, pre) else
                           "future" if re.search(FUT_TRIG, pre) else "current")
            ents.append(dict(text=m.group(0), type=typ, evidence=sent.strip()[:200],
                             negated=bool(re.search(NEG_TRIG, pre)), certainty=certainty,
                             temporality=temporality, related_test=None, value=None))
    return dict(extraction={"entities": ents}, valid_json=True, valid_schema=True,
                latency_s=time.perf_counter() - t0)

preds = {}
b0 = [dict(unit_id=u, **rule_extract(t)) for u, t in zip(test_units.unit_id, test_units.text)]
preds["B0_rules"] = (pd.DataFrame(b0), 0.0)

### 9.2 Run the LLM systems

In [ ]:
# ════════ ✅ RUN · STEP 22 · run/load B2, B1, FT  (+ FIX C re-parse) ════════
# 9.2 Run the LLM systems on the test set (cache-aware; includes FIX C re-parse)
def load_cached(path):
    out = {}
    if Path(path).exists():
        for line in open(path):
            try:
                d = json.loads(line)
                out[d["unit_id"]] = d
            except json.JSONDecodeError:
                pass
    return out

def reparse_df(path):
    # FIX C: re-parse saved raw outputs with the forgiving parser (no model calls)
    cache = load_cached(path)
    rows = []
    for u in test_units.unit_id:
        d = dict(cache[u])
        if d.get("raw") is not None:
            ext, ok_json, ok_schema = parse_json(d["raw"])
            d.update(extraction=ext.model_dump() if ext else None,
                     valid_json=ok_json, valid_schema=ok_schema)
        rows.append(d)
    return pd.DataFrame(rows)

test_ids = set(test_units.unit_id)

# ---------- B2: Qwen3.6-35B (vLLM) ----------
b2 = run_batch(test_units, OUT_DIR / "preds/B2_qwen36_zeroshot.jsonl", constrained=True, workers=6)
preds["B2_qwen3.6-35B_zeroshot"] = (b2, float("nan"))

# ---------- B1: Qwen2.5-3B, no fine-tuning ----------
b1_path = OUT_DIR / "preds/B1_qwen25_3b_zeroshot.jsonl"
b1_mem = float("nan")
if len(test_ids & set(load_cached(b1_path))) < len(test_ids):
    hf_base = HFExtractor(STUDENT_BASE)
    _, b1_mem = run_hf(hf_base, test_units, b1_path)
    del hf_base; torch.cuda.empty_cache()
preds["B1_qwen2.5-3B_zeroshot"] = (reparse_df(b1_path), b1_mem)

# ---------- FT: Qwen2.5-3B + LoRA ----------
ft_path = OUT_DIR / "preds/FT_qwen25_3b_lora.jsonl"
if (ADAPTER_DIR / "adapter_config.json").exists():
    ft_mem = float("nan")
    if len(test_ids & set(load_cached(ft_path))) < len(test_ids):
        hf_ft = HFExtractor(STUDENT_BASE, adapter=ADAPTER_DIR)
        _, ft_mem = run_hf(hf_ft, test_units, ft_path)
        del hf_ft; torch.cuda.empty_cache()
    preds["FT_qwen2.5-3B_LoRA"] = (reparse_df(ft_path), ft_mem)
else:
    print("⚠️ FT skipped: no LoRA adapter yet. Set RUN_TRAINING = True (step 1) and run step 17 first.")

print("\n=== Systems ready for evaluation ===")
for name, (df, mem) in preds.items():
    n_ent = df.extraction.map(lambda x: len(x["entities"]) if isinstance(x, dict) else 0).sum() if "extraction" in df else 0
    print(f"{name:28s} chunks={len(df):4d}  entities={n_ent:6d}  peak_mem_GB={mem}")


## 10. Evaluation (Objective 5)

**Primary metric:** micro-averaged entity F1 (relaxed match = same type and token-Jaccard ≥ 0.5; strict = exact normalised text).

**Secondary metrics:**
* precision, recall, macro-F1 across entity types; per-type P/R/F1
* **attribute accuracy** (negation, certainty, temporality) on matched entities
* **JSON validity** and **schema validity** rates
* **evidence support rate** and **unsupported extraction rate** (fraction of predicted entities whose evidence is not found in the note)
* **latency** (mean, p95 seconds per chunk) and **peak GPU memory**
* **95 % bootstrap confidence intervals** (1 000 resamples of test units)

In [ ]:
# ════════ ✅ RUN · STEP 23 · evaluation / reports (CPU, minutes) ════════
TYPES = ["PROBLEM", "MEDICATION", "TEST", "RESULT"]
ATTRS = ["negated", "certainty", "temporality"]

def unit_counts(ref, pred, mode):
    # Per-unit, per-type TP/FP/FN + attribute agreement on matched pairs
    c = {t: [0, 0, 0] for t in TYPES}; attr_hits = {a: [0, 0] for a in ATTRS}
    pairs, mg, mp = match_entities(ref, pred, mode=mode)
    for i, j in pairs:
        c[ref[i]["type"]][0] += 1
        for a in ATTRS:
            attr_hits[a][0] += int(ref[i][a] == pred[j].get(a)); attr_hits[a][1] += 1
    for i in mg: c[ref[i]["type"]][2] += 1
    for j in mp:
        if pred[j]["type"] in c: c[pred[j]["type"]][1] += 1
    return c, attr_hits

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def aggregate(per_unit):
    tot = {t: np.sum([u[0][t] for u in per_unit], axis=0) for t in TYPES}
    TP, FP, FN = np.sum(list(tot.values()), axis=0)
    p, r, f = prf(TP, FP, FN)
    macro = np.mean([prf(*tot[t])[2] for t in TYPES if tot[t].sum() > 0])
    attrs = {a: (sum(u[1][a][0] for u in per_unit) / max(sum(u[1][a][1] for u in per_unit), 1)) for a in ATTRS}
    return dict(P=p, R=r, micro_F1=f, macro_F1=macro, **{f"acc_{a}": v for a, v in attrs.items()}), tot

def bootstrap_ci(per_unit, key="micro_F1", B=1000, seed=SEED):
    rng = np.random.default_rng(seed); n = len(per_unit); vals = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        vals.append(aggregate([per_unit[i] for i in idx])[0][key])
    return np.percentile(vals, [2.5, 97.5])

def evaluate(name, df, peak_mem, mode="relaxed"):
    text_of = dict(zip(test_units.unit_id, test_units.text))
    per_unit, n_pred, n_sup = [], 0, 0
    for r in df.itertuples():
        pred = (r.extraction or {"entities": []})["entities"]
        for e in pred:
            n_pred += 1; n_sup += is_supported(e, text_of[r.unit_id])
        per_unit.append(unit_counts(REF[r.unit_id]["entities"], pred, mode))
    m, tot = aggregate(per_unit)
    lo, hi = bootstrap_ci(per_unit)
    lat = df.latency_s.dropna()
    row = dict(system=name, match=mode, **{k: round(v, 3) for k, v in m.items()},
               micro_F1_95CI=f"[{lo:.3f}, {hi:.3f}]",
               json_valid=round(df.valid_json.mean(), 3), schema_valid=round(df.valid_schema.mean(), 3),
               unsupported_rate=round(1 - n_sup / max(n_pred, 1), 3), n_pred=n_pred,
               latency_mean_s=round(lat.mean(), 3), latency_p95_s=round(lat.quantile(0.95), 3),
               peak_gpu_mem_GB=round(peak_mem, 2) if peak_mem == peak_mem else None)
    per_type = pd.DataFrame({t: dict(zip(["P", "R", "F1"], prf(*tot[t])), support=int(tot[t][0] + tot[t][2]))
                             for t in TYPES}).T.assign(system=name)
    return row, per_type, per_unit

results, per_type_all, per_unit_store = [], [], {}
for name, (df, mem) in preds.items():
    for mode in ["relaxed", "strict"]:
        row, pt, pu = evaluate(name, df, mem, mode)
        results.append(row)
        if mode == "relaxed": per_type_all.append(pt); per_unit_store[name] = pu

res_df = pd.DataFrame(results)
res_df.to_csv(OUT_DIR / "results/main_results.csv", index=False)
res_df[res_df.match == "relaxed"].set_index("system")

In [ ]:
# ════════ ✅ RUN · STEP 24 · evaluation / reports (CPU, minutes) ════════
per_type_df = pd.concat(per_type_all).reset_index().rename(columns={"index": "type"})
per_type_df.to_csv(OUT_DIR / "results/per_type_results.csv", index=False)
per_type_df.pivot(index="type", columns="system", values="F1").round(3)

In [ ]:
# ════════ ✅ RUN · STEP 25 · evaluation / reports (CPU, minutes) ════════
rel = res_df[res_df.match == "relaxed"].set_index("system")
ci = rel.micro_F1_95CI.str.strip("[]").str.split(",", expand=True).astype(float)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(rel.index, rel.micro_F1, xerr=[rel.micro_F1 - ci[0], ci[1] - rel.micro_F1],
        color="#4C72B0", capsize=4)
ax.set_xlabel("Micro-F1 (relaxed) with 95% bootstrap CI"); ax.set_xlim(0, 1)
ax.set_title("Clinical information extraction — held-out test patients")
plt.tight_layout(); plt.savefig(OUT_DIR / "figures/micro_f1.png", dpi=200); plt.show()

### 10.1 Paired significance test (FT vs. base model)
Paired bootstrap over test units: the proportion of resamples in which the fine-tuned model does **not** outperform the baseline approximates a one-sided *p*-value.

In [ ]:
# ════════ ✅ RUN · STEP 26 · evaluation / reports (CPU, minutes) ════════
def paired_bootstrap(pu_a, pu_b, B=2000, seed=SEED):
    rng = np.random.default_rng(seed); n = len(pu_a); wins = 0; deltas = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        fa = aggregate([pu_a[i] for i in idx])[0]["micro_F1"]
        fb = aggregate([pu_b[i] for i in idx])[0]["micro_F1"]
        deltas.append(fa - fb); wins += fa > fb
    return dict(delta_mean=np.mean(deltas), CI95=np.percentile(deltas, [2.5, 97.5]).round(3), p_value=1 - wins / B)

if "FT_qwen2.5-3B_LoRA" in per_unit_store:
    for base in ["B1_qwen2.5-3B_zeroshot", "B0_rules", "B2_qwen3.6-35B_zeroshot"]:
        if base in per_unit_store:
            print(f"FT vs {base}:", paired_bootstrap(per_unit_store["FT_qwen2.5-3B_LoRA"], per_unit_store[base]))

### 10.2 Generalisation across note categories
Performance stratified by note type (discharge vs. physician vs. nursing) on held-out patients.

In [ ]:
# ════════ ✅ RUN · STEP 27 · evaluation / reports (CPU, minutes) ════════
cat_of = dict(zip(test_units.unit_id, test_units.category))
rows = []
for name, (df, _) in preds.items():
    df = df.assign(category=df.unit_id.map(cat_of))
    for cat, g in df.groupby("category"):
        pu = [unit_counts(REF[r.unit_id]["entities"], (r.extraction or {"entities": []})["entities"], "relaxed")
              for r in g.itertuples()]
        rows.append(dict(system=name, category=cat, n=len(g), micro_F1=round(aggregate(pu)[0]["micro_F1"], 3)))
pd.DataFrame(rows).pivot(index="category", columns="system", values="micro_F1")

## 11. Error analysis
Categorise errors for the fine-tuned model (or the best available system): false positives, false negatives, type confusions, and attribute errors — with special attention to **negation** and **uncertainty**, as stated in the expected outcomes.

In [ ]:
# ════════ ✅ RUN · STEP 28 · evaluation / reports (CPU, minutes) ════════
focus = "FT_qwen2.5-3B_LoRA" if "FT_qwen2.5-3B_LoRA" in preds else next(iter(preds))
df_f = preds[focus][0]
err_rows = []
for r in df_f.itertuples():
    ref = REF[r.unit_id]["entities"]; pred = (r.extraction or {"entities": []})["entities"]
    pairs, mg, mp = match_entities(ref, pred)
    for i in mg: err_rows.append(dict(unit_id=r.unit_id, error="FN", type=ref[i]["type"], text=ref[i]["text"], detail=""))
    for j in mp:
        # type confusion if same text exists in gold with different type
        conf = [g["type"] for g in ref if norm_text(g["text"]) == norm_text(pred[j]["text"])]
        err_rows.append(dict(unit_id=r.unit_id, error="TYPE_CONFUSION" if conf else "FP",
                             type=pred[j]["type"], text=pred[j]["text"], detail=f"gold={conf[0]}" if conf else ""))
    for i, j in pairs:
        for a in ATTRS:
            if ref[i][a] != pred[j].get(a):
                err_rows.append(dict(unit_id=r.unit_id, error=f"ATTR_{a}", type=ref[i]["type"], text=ref[i]["text"],
                                     detail=f"gold={ref[i][a]} pred={pred[j].get(a)}"))
errors = pd.DataFrame(err_rows)
errors.to_csv(OUT_DIR / f"results/errors_{focus}.csv", index=False)
print("System:", focus)
errors.groupby(["error", "type"]).size().unstack(fill_value=0)

In [ ]:
# ════════ ✅ RUN · STEP 29 · evaluation / reports (CPU, minutes) ════════
# Most frequent attribute errors (e.g. missed negations)
errors[errors.error.str.startswith("ATTR")].groupby(["error", "detail"]).size().sort_values(ascending=False).head(15)

In [ ]:
# ════════ ✅ RUN · STEP 30 · evaluation / reports (CPU, minutes) ════════
# Inspect examples of unsupported (hallucinated) predictions from each LLM system
text_of = dict(zip(test_units.unit_id, test_units.text))
for name, (df, _) in preds.items():
    if name.startswith("B0"): continue
    bad = [(r.unit_id, e["text"], e["evidence"]) for r in df.itertuples()
           for e in (r.extraction or {"entities": []})["entities"] if not is_supported(e, text_of[r.unit_id])]
    print(f"\n=== {name}: {len(bad)} unsupported ===")
    for b in bad[:5]: print("  ", b)

## 12. Reproducibility record
Saves software versions, configuration, data-split hashes and model identifiers alongside the results.

In [ ]:
# ════════ ✅ RUN · STEP 31 · evaluation / reports (CPU, minutes) ════════
import transformers, peft, trl, sklearn
repro = dict(
    timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), platform=platform.platform(), python=platform.python_version(),
    torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__, trl=trl.__version__,
    duckdb=duckdb.__version__, sklearn=sklearn.__version__,
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    teacher_model=MODEL, student_base=STUDENT_BASE, seed=SEED,
    note_categories=NOTE_CATEGORIES, max_chunk_chars=MAX_CHUNK_CHARS, n_annotation_units=N_ANNOTATION_UNITS,
    split_patients={sp: int((units if "units" in dir() else ann).query("split == @sp").subject_id.nunique()) for sp in ["train", "val", "test"]},
    test_unit_hash=hashlib.sha1(",".join(sorted(test_units.unit_id)).encode()).hexdigest(),
    gold_test_units=int(sum(u in gold for u in test_units.unit_id)),
)
json.dump(repro, open(OUT_DIR / "results/reproducibility.json", "w"), indent=2)
repro

## 13. Limitations & next steps
* **Gold standard size** — confidence intervals will be wide until enough test units are human-annotated; prioritise test-set gold annotation.
* **Silver-label bias** — training on teacher outputs can transfer its systematic errors; compare an FT model trained on gold-only train data as an ablation.
* **Ablations to consider** — LoRA rank (8/16/64), QLoRA vs. bf16 LoRA, training-set size learning curve, constrained vs. unconstrained decoding for the FT model.
* **External validity** — MIMIC-III is a single-centre US ICU dataset; generalisation to Indian hospital records would require separate validation.
* The system is a **research prototype** and must not be used for clinical decision-making.